In [1]:
# ============================================================
# AI USE CASE 2: BASELINE MONTHLY DEMAND FORECAST
# Forecasts monthly orders and net sales for the next 3 months.
# Same rules as SQL / EDA / Power BI: window 2015-01-01 to 2017-09-30.
# Outputs: demand_forecast.csv (for Power BI) and forecast_backtest.csv
# ============================================================
# pip install pandas numpy openpyxl

from pathlib import Path

import numpy as np
import pandas as pd

WINDOW_START, WINDOW_END = pd.Timestamp("2015-01-01"), pd.Timestamp("2017-09-30")
FILE_PATH = Path("DataCo.xlsx")
BACKTEST_MONTHS = 12      # test each method on the last 12 months
HORIZON = 3               # forecast the next 3 months

# ------------------------------------------------------------
# 1. LOAD AND CLEAN (same steps as the other scripts)
# ------------------------------------------------------------
df = pd.read_excel(FILE_PATH, engine="openpyxl")
df.columns = (df.columns.astype(str).str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))


def parse_mixed_date(value):
    """Text cells are M/D/YYYY; real Excel dates have day and month swapped."""
    if isinstance(value, str):
        return pd.to_datetime(value.strip(), format="%m/%d/%Y %H:%M")
    ts = pd.Timestamp(value)
    return pd.Timestamp(year=ts.year, month=ts.day, day=ts.month,
                        hour=ts.hour, minute=ts.minute)


df["order_ts"] = df["order_date_dateorders"].map(parse_mixed_date)
df["ship_ts"] = df["shipping_date_dateorders"].map(parse_mixed_date)
gap = (df["ship_ts"].dt.normalize() - df["order_ts"].dt.normalize()).dt.days
assert (gap == df["days_for_shipping_real"]).all(), "Date parsing is wrong"
df["month"] = df["order_ts"].dt.to_period("M")

in_window = df["order_ts"].dt.normalize().between(WINDOW_START, WINDOW_END)
monthly_all = (df.groupby("month")
                 .agg(orders=("order_id", "nunique"), net_sales=("order_item_total", "sum")))
monthly = monthly_all[monthly_all.index <= pd.Period("2017-09")]
print(f"Months in window: {len(monthly)} "
      f"({monthly.index.min()} to {monthly.index.max()})")

# ------------------------------------------------------------
# 2. FORECASTING METHODS (all simple on purpose)
# ------------------------------------------------------------
def f_naive(h):      return h.iloc[-1]
def f_mean_all(h):   return h.mean()
def f_mean_3(h):     return h.iloc[-3:].mean()
def f_mean_6(h):     return h.iloc[-6:].mean()
def f_linear(h):
    slope, intercept = np.polyfit(np.arange(len(h)), h.values, 1)
    return slope * len(h) + intercept

METHODS = {"Last month (naive)": f_naive, "Average of all months": f_mean_all,
           "Average of last 3 months": f_mean_3, "Average of last 6 months": f_mean_6,
           "Linear trend": f_linear}

# ------------------------------------------------------------
# 3. ROLLING BACKTEST: forecast each of the last 12 months using only
#    earlier months, then compare with what really happened.
# ------------------------------------------------------------
rows, errors = [], {}
for metric in ["orders", "net_sales"]:
    series = monthly[metric]
    start = len(series) - BACKTEST_MONTHS
    for name, func in METHODS.items():
        pct_err = []
        for t in range(start, len(series)):
            pred = func(series.iloc[:t])
            pct_err.append((series.iloc[t] - pred) / series.iloc[t] * 100)
        errors[(metric, name)] = np.array(pct_err)
        rows.append({"metric": metric, "method": name,
                     "MAPE_pct": np.abs(pct_err).mean(),
                     "bias_pct": np.mean(pct_err)})
backtest = pd.DataFrame(rows)
print("\n--- BACKTEST: last 12 months, one-month-ahead (lower MAPE is better) ---")
print(backtest.pivot(index="method", columns="metric", values="MAPE_pct").round(2))
backtest.to_csv("forecast_backtest.csv", index=False)

variation = (monthly.std() / monthly.mean() * 100).round(2)
print("\nMonth-to-month variation (std / mean, %):")
print(variation.to_string())

# ------------------------------------------------------------
# 4. FORECAST THE NEXT 3 MONTHS WITH THE BEST METHOD PER METRIC
#    Range = forecast +/- 1.96 x standard deviation of backtest errors.
# ------------------------------------------------------------
future = pd.period_range(monthly.index.max() + 1, periods=HORIZON, freq="M")
out = []
print("\n--- FORECAST ---")
for metric in ["orders", "net_sales"]:
    best = backtest[backtest.metric == metric].sort_values("MAPE_pct").iloc[0]["method"]
    point = METHODS[best](monthly[metric])
    spread = 1.96 * errors[(metric, best)].std() / 100 * point
    print(f"{metric}: best method = {best} | forecast per month = {point:,.0f} "
          f"(range {point - spread:,.0f} to {point + spread:,.0f})")
    for m in future:
        actual = monthly_all[metric].get(m, np.nan)
        out.append({"month": m.to_timestamp(), "metric": metric, "method": best,
                    "forecast": round(point, 2), "lower": round(point - spread, 2),
                    "upper": round(point + spread, 2), "actual_if_available": actual})
result = pd.DataFrame(out)

# ------------------------------------------------------------
# 5. REALITY CHECK: compare with the months after the window
#    (data exists to Jan 2018 but has a different structure).
# ------------------------------------------------------------
print("\n--- REALITY CHECK: forecast vs actual AFTER the window (outside analysis window) ---")
check = result.copy()
check["actual_if_available"] = check["actual_if_available"].round(0)
print(check[["month", "metric", "forecast", "lower", "upper", "actual_if_available"]]
      .to_string(index=False))
print("\nActual months after the window:")
print(monthly_all[monthly_all.index > pd.Period("2017-09")].round(0).to_string())

result.to_csv("demand_forecast.csv", index=False)
print("\nSaved demand_forecast.csv and forecast_backtest.csv")

# ------------------------------------------------------------
# INTERPRETATION
# - Monthly volumes barely move, so every simple method is within a few
#   percent. A trend or machine-learning model would add complexity and no
#   accuracy: the honest forecast is "about the same as recent months".
# - Do not forecast by market or category: markets sell in separate blocks of
#   months and many categories are missing in some months.
# - Months after Sep 2017 have a different structure (see reality check), so
#   they are shown for comparison only and are not used for training.
# ------------------------------------------------------------

Months in window: 33 (2015-01 to 2017-09)

--- BACKTEST: last 12 months, one-month-ahead (lower MAPE is better) ---
metric                    net_sales  orders
method                                     
Average of all months          3.58    2.12
Average of last 3 months       2.98    2.69
Average of last 6 months       3.06    2.52
Last month (naive)             3.29    3.71
Linear trend                   3.10    2.23

Month-to-month variation (std / mean, %):
orders       2.87
net_sales    3.86

--- FORECAST ---
orders: best method = Average of all months | forecast per month = 1,738 (range 1,639 to 1,836)
net_sales: best method = Average of last 3 months | forecast per month = 1,005,348 (range 941,963 to 1,068,732)

--- REALITY CHECK: forecast vs actual AFTER the window (outside analysis window) ---
     month    metric   forecast     lower      upper  actual_if_available
2017-10-01    orders    1737.85   1639.40    1836.30               2101.0
2017-11-01    orders    1737.85   163